In [ ]:
!pip install openai tiktoken

In [ ]:
import getpass, openai, os
client = getpass.getpass(prompt="OPENAI - KEY: ")
openai.apikey = client
os.environ["OPENAI_API_KEY"] = client


In [ ]:
from openai import OpenAI
client = OpenAI()

In [ ]:
import requests
import json

movieSearchResults = []

for page in [1, 2]:
    omdbApiRequest = requests.get(f"https://www.omdbapi.com/?s=saw&page={page}&apikey=10ebe91b")
    movieResultsJson = json.loads(omdbApiRequest.text)
    movieSearchResults += movieResultsJson["Search"]

movieDetails = []

for movie in movieSearchResults:
    movieId = movie["imdbID"]
    movieSearch = requests.get(f"https://www.omdbapi.com/?i={movieId}&apikey=10ebe91b")
    movieDetails += [json.loads(movieSearch.text)]

In [ ]:
clases = [pelicula["Type"] for pelicula in movieSearchResults]

# Crear un conjunto a partir de la lista de clases
clases_unicas = set(clases)
clases_unicas

In [ ]:
moviesString = ""
for mediaType in list(clases_unicas):
    moviesString += f"\n{mediaType}:\n"
    for movie in movieDetails:
        if movie["Type"] == mediaType:
            moviesString += f"{movie['Title']}\n"

print(moviesString)

In [ ]:
def createModeration(query):
    moderation = client.moderations.create(input = query)
    results = moderation.results
    categories = zip(results[0].categories, results[0].category_scores)
    flaggedStatus = results[0].flagged
    return categories, flaggedStatus

In [ ]:

system_message = f"""
You will be provided with queries about media. \
The user query will be delimited with \
{DELIMITER} characters.

Output a python list of objects, where each object has \
the following format:
    'Type': <one of 'series', 'game', 'movie'>,
OR
    'Titles': <a list of media titles that must \
    be found in the allowed titles below>

Where the types of media and the titles must be found in \
the customer service query.
Whenever a title is mentioned, your output must associate it with its \
respective type as in the allowed titles list below.
If no titles or types are found, output an \
empty list.


Allowed media:

{moviesString}

First, check whether the title or media type are explicitly in the query.

If not, check whether they are implied.



Only output the list of objects, with nothing else.
"""

user_query = """
What can you tell me about the saw movies?
"""



In [ ]:
DELIMITER = "####"

def getCompletionFromMessages(
        query,
        messages,
        model = "gpt-4",
        temperature = 0,
        delimiter = DELIMITER
):
    query = f"{DELIMITER}{query}{DELIMITER}"
    messages += [{"role": "user", "content": query}]
    response = client.chat.completions.create(
        messages = messages,
        temperature = temperature,
        model = model
    )
    responseContent = response.choices[0].message.content
    messages += [{"content": responseContent, "role": "assistant"}]
    return messages

In [ ]:
messages =  [{'role':'system', 'content': system_message}]

messages = getCompletionFromMessages(user_query, messages)

messages[-1]["content"]

In [ ]:
mediaResultsData = json.loads(messages[-1]["content"].replace("'", "\""))

In [ ]:
def get_movie_by_name(title):
    return [movie for movie in movieDetails if movie["Title"] == title]

def get_movies_by_type(mediaType):
    return [movie for movie in movieDetails if movie["Type"] == mediaType]

In [ ]:
def generate_output_string(data_list):
    output_string = ""

    if data_list is None:
        return output_string

    for data in data_list:
        try:
            if "Titles" in data:
                titles_list = data["Titles"]
                for media_name in titles_list:
                    media = get_movie_by_name(media_name)
                    if media:
                        output_string += json.dumps(media, indent=4) + "\n"
                    else:
                        print(f"Error: Media '{media_name}' not found")
            elif "Type" in data:
                type_name = data["Type"]
                type_media = get_movies_by_type(type_name)
                for media in type_media:
                    output_string += json.dumps(media, indent=4) + "\n"
            else:
                print("Error: Invalid object format")
        except Exception as e:
            print(f"Error: {e}")

    return output_string

In [ ]:
print(generate_output_string(mediaResultsData))

In [ ]:
infoParsingSystemPrompt = f"""
You are a customer service assistant for a \
large electronic store. \
Respond in a friendly and helpful tone, \
with very concise answers. \
Make sure to ask the user relevant follow up questions.

You are a helpful assistant tasked with giving information about media.
You will be given a user query, delimited by {DELIMITER}, and data about \
media in JSON format. Use the data provided to answer the query
"""

def getCompletionFromMessages(
        messages,
        model = "gpt-4",
        temperature = 0
):
    response = client.chat.completions.create(
        messages = messages,
        temperature = temperature,
        model = model
    )
    responseContent = response.choices[0].message.content
    messages += [{"content": responseContent, "role": "assistant"}]
    return messages

def getAssistantMediaInfo(query, messages, delimiter = DELIMITER):
    processedQuery = f"{delimiter}{query}{delimiter}"
    mediaSearchMessages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": processedQuery}
    ]
    searchResultsString = getCompletionFromMessages(mediaSearchMessages)[-1]["content"]
    resultsData = json.loads(searchResultsString.replace("'", "\""))
    mediaInfoString = generate_output_string(mediaResultsData)
    processedQuery += f"\n{mediaInfoString}"
    messages += [{"role": "user", "content": processedQuery}]
    messages = getCompletionFromMessages(messages)
    print(messages[-1]["content"])
    return messages

In [ ]:
messages = [{"role": "system", "content": infoParsingSystemPrompt}]

messages = getAssistantMediaInfo(user_query, messages)

In [ ]:
categories, flagged = createModeration(messages[-1]["content"])

for category in categories:
    print(category)

print("FLAGGED BY MODERATION:", flagged)

In [ ]:
messages[-1]["content"]